# Interactive Filter

## What you'll learn

- Load artifacts and their metrics into an InteractiveFilter
- Explore metric distributions with wide and tidy DataFrames
- Set filter criteria and inspect pass rates
- Visualize distributions with threshold overlay plots
- Commit the filter result as a pipeline step

The `Filter` operation requires you to specify criteria upfront.
**InteractiveFilter** flips the workflow: load metrics first, explore
distributions, then decide on thresholds — all in a notebook. When
you're satisfied, `.commit()` writes the filter as a proper pipeline
step with full provenance.

**Prerequisites:** [Metrics and Filtering](../pipeline-patterns/03-metrics-and-filtering.ipynb),
[Exploring Results](../getting-started/02-exploring-results.ipynb).  
**Time:** ~15 min.  
**GPU required:** No.

In [ ]:
from __future__ import annotations

from artisan.operations.curator.interactive_filter import InteractiveFilter
from artisan.operations.examples import (
    DataGenerator,
    DataTransformer,
    MetricCalculator,
)
from artisan.orchestration import Backend, PipelineManager
from artisan.utils import tutorial_setup
from artisan.utils.logging import configure_logging
from artisan.visualization import build_macro_graph, inspect_pipeline

configure_logging()

In [ ]:
env = tutorial_setup("interactive_filter")

## Build a pipeline with metrics

InteractiveFilter needs a pipeline that has already computed metrics.
The pipeline below generates data, transforms it, and computes metrics
— giving us numeric distributions to explore and filter on.

In [ ]:
pipeline = PipelineManager.create(
    name="interactive_filter_tutorial",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)
output = pipeline.output

pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 8, "seed": 42},
    backend=Backend.LOCAL,
)
pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("generate", "datasets")},
    backend=Backend.LOCAL,
)
pipeline.run(
    operation=MetricCalculator,
    name="score",
    inputs={"dataset": output("transform", "dataset")},
    backend=Backend.LOCAL,
)
pipeline.finalize()
inspect_pipeline(env.delta_root)

## Load artifacts into the interactive filter

Create an `InteractiveFilter` pointed at the delta root, then call
`.load()` to pull in artifacts and their derived metrics. The
`step_numbers` argument controls which primary artifacts to load —
here we load the transformed datasets from step 1.

In [ ]:
filt = InteractiveFilter(env.delta_root)
filt.load(step_numbers=[1])

## Explore metric distributions

InteractiveFilter provides two DataFrame views of the loaded data.
**`wide_df`** has one row per artifact with metrics as columns — good
for scanning values at a glance. **`tidy_df`** has one row per
(artifact, metric_name) pair — good for aggregation and plotting.

In [ ]:
filt.wide_df

In [ ]:
filt.tidy_df

The **wide DataFrame** columns are metric names drawn from the metric
artifacts. Nested values are dot-separated (e.g. `distribution.median`,
`summary.cv`). Each row represents one artifact from the loaded step.

The **tidy DataFrame** has columns: `artifact_id`, `step_number`,
`step_name`, `metric_name`, and `metric_value`. This long format is
convenient for grouping, aggregation, and plotting libraries that
expect one observation per row.

## Set filter criteria

Now set filter criteria. Each criterion specifies a metric column name,
a comparison operator (`gt`, `ge`, `lt`, `le`, `eq`, `ne`), and a
threshold value. The metric name must match a column in `wide_df`.

In [ ]:
filt.set_criteria(
    [
        {"metric": "distribution.median", "operator": "gt", "value": 0.3},
        {"metric": "summary.cv", "operator": "lt", "value": 0.8},
    ]
)

## Inspect pass rates

Call `.summary()` to see per-criterion pass rates and a cumulative
funnel showing how many artifacts survive each successive filter.

In [ ]:
filt.summary()

The **criteria table** shows each criterion with its threshold, pass
count, pass rate, and distribution statistics (min, median, max). The
**funnel table** shows how the artifact count drops as each criterion
is applied cumulatively — this helps you spot which criterion is the
most restrictive.

## Visualize distributions

The `.plot()` method renders one histogram per criterion with a red
dashed threshold line. This helps you visually assess where your
threshold sits relative to the distribution.

In [ ]:
filt.plot()

## Iterate on thresholds

If the pass rates aren't what you want, adjust the criteria and
re-check. This is the core advantage over `Filter`: you can iterate
on thresholds without re-running the pipeline.

In [ ]:
filt.set_criteria(
    [
        {"metric": "distribution.median", "operator": "gt", "value": 0.4},
    ]
)
filt.summary()

## Commit the filter as a pipeline step

When you're satisfied with the thresholds, call `.commit()` to write
the filter as a pipeline step. This creates step, execution, and
provenance records in the delta store — making the filter result
available to downstream steps via `result.output("passthrough")`.

In [ ]:
result = filt.commit()
print(f"Committed as step {result.step_number}")
print(f"  {result.succeeded_count} / {result.total_count} artifacts passed")

## Inspect the pipeline after commit

The committed filter now appears in the pipeline overview and the
macro graph.

In [ ]:
inspect_pipeline(env.delta_root)

In [ ]:
build_macro_graph(env.delta_root)

The macro graph shows the interactive filter as a regular pipeline
step. Downstream operations can consume `result.output("passthrough")`
exactly like a `Filter` step — the only difference is that the
thresholds were chosen interactively rather than specified upfront.

## Summary

| Concept | API | Purpose |
|---------|-----|--------|
| Load | `InteractiveFilter(delta_root).load(step_numbers=[...])` | Pull artifacts and derived metrics |
| Wide view | `.wide_df` | One row per artifact, metrics as columns |
| Tidy view | `.tidy_df` | One row per (artifact, metric), good for aggregation |
| Criteria | `.set_criteria([{...}])` | Define filter thresholds |
| Summary | `.summary()` | Per-criterion stats and cumulative funnel |
| Plot | `.plot()` | Histograms with threshold lines |
| Commit | `.commit()` | Write filter as a pipeline step |

**Key takeaway:** InteractiveFilter bridges exploratory analysis and
pipeline execution. Explore metrics freely in a notebook, then commit
your thresholds as a tracked, reproducible pipeline step with full
provenance.

## Next steps

- [Timing Analysis](04-timing-analysis.ipynb) — Profile step and execution performance
- [Provenance Graphs](01-provenance-graphs.ipynb) — Visualize the filter step in context
- [Metrics and Filtering](../pipeline-patterns/03-metrics-and-filtering.ipynb) — Automated filtering with upfront criteria